# LCE Review Pipeline — 03 · Genie Agent (Gold → Genie space)

Turns `reviews_gold` into a natural-language analytics surface:

1. **`reviews_gold_genie`** — a flattened, analytics-friendly table built from `reviews_gold`.
   The app's array/struct columns (`classification`, `product`, `product_issues`) are
   exploded/denormalized into flat columns and one row **per review × category**, which
   is the grain Genie reasons about best. Column comments are written in-table so Genie
   picks up the metadata automatically.
2. **A new Genie space** wired to that table, with column descriptions/synonyms,
   text instructions, curated example-question SQL, and benchmark questions.

| Layer | Object | Grain | Contents |
|-------|--------|-------|----------|
| Gold  | `reviews_gold`       | 1 row / review | app serving table (native arrays) |
| Genie | `reviews_gold_genie` | 1 row / review × category | flattened for NL analytics |

> **Grain note:** because a review is exploded to one row per mentioned category, review-level
> counts/averages must dedupe. The table carries `is_primary_category_row` (TRUE on exactly one
> row per review) so `WHERE is_primary_category_row` gives back review grain. This is documented
> in the column comment and the Genie instructions.

> **Env:** `catalog`/`schema` widgets (prod `ioc_sandbox.ai_strategy`). `warehouse_id` powers the
> Genie space. `genie_parent_path` is the workspace folder the space is created under.

In [ ]:
# --- Parameters (widgets) ---------------------------------------------------
dbutils.widgets.text("catalog", "ioc_sandbox", "Catalog")
dbutils.widgets.text("schema", "ai_strategy", "Schema")
dbutils.widgets.text("gold_table", "reviews_gold", "Source gold table")
dbutils.widgets.text("genie_table", "reviews_gold_genie", "Genie analytics table")
dbutils.widgets.text("warehouse_id", "610ea94e1066f95b", "SQL warehouse id (for the Genie space)")
dbutils.widgets.text("genie_parent_path", "", "Genie space parent folder (blank = current user home)")
dbutils.widgets.text("space_title", "LCE Guest Reviews", "Genie space title")

CATALOG    = dbutils.widgets.get("catalog").strip()
SCHEMA     = dbutils.widgets.get("schema").strip()
GOLD       = f"{CATALOG}.{SCHEMA}.{dbutils.widgets.get('gold_table').strip()}"
GENIE_TBL  = f"{CATALOG}.{SCHEMA}.{dbutils.widgets.get('genie_table').strip()}"
WAREHOUSE  = dbutils.widgets.get("warehouse_id").strip()
PARENT     = dbutils.widgets.get("genie_parent_path").strip()
TITLE      = dbutils.widgets.get("space_title").strip()

print(f"{GOLD}  ->  {GENIE_TBL}")
print(f"warehouse: {WAREHOUSE}")

## 1 · Build `reviews_gold_genie`

Flatten the arrays into a wide, self-describing, review×category-grain table:
- `city` / `state` split out of `location_name` ("City, ST").
- `review_date` as a real DATE.
- `category` + `category_rating`: the exploded classification aspect and its matching 1–5 rating.
- `is_primary_category_row`: TRUE on one row per review, so review-level aggregates can dedupe.
- `products` / `product_issues_str`: human-readable delimited strings (secondary analytics axis).
- `n_products` / `n_product_issues`: simple counts for filtering.

In [ ]:
GENIE_SQL = f"""
CREATE OR REPLACE TABLE {GENIE_TBL} AS
WITH base AS (
  SELECT
    locationId,
    location_name,
    split(location_name, ', ')[0]              AS city,
    element_at(split(location_name, ', '), -1) AS state,
    to_date(date)                              AS review_date,
    rating,
    sentiment,
    directoryType                              AS review_source,
    coalesce(comment, review_text)             AS review_text,
    speed_rating, cleanliness_rating, order_accuracy_rating, quality_rating, service_rating,
    classification,
    coalesce(product, array())                 AS product,
    coalesce(product_issues, array())          AS product_issues,
    -- one stable primary row per review (first category by posexplode ordinal below)
    monotonically_increasing_id()              AS _review_uid
  FROM {GOLD}
)
SELECT
  b.locationId,
  b.location_name,
  b.city,
  b.state,
  b.review_date,
  b.rating,
  b.sentiment,
  b.review_source,
  b.review_text,
  c.category,
  CASE c.category
    WHEN 'Speed'          THEN b.speed_rating
    WHEN 'Cleanliness'    THEN b.cleanliness_rating
    WHEN 'Order Accuracy' THEN b.order_accuracy_rating
    WHEN 'Quality'        THEN b.quality_rating
    WHEN 'Service'        THEN b.service_rating
  END                                          AS category_rating,
  -- pos=0 marks the first category of a classified review; pos IS NULL is the single
  -- OUTER row for a review with no classification. Both are the review's one primary row.
  (c.pos = 0 OR c.pos IS NULL)                 AS is_primary_category_row,
  concat_ws(', ', b.product)                   AS products,
  size(b.product)                              AS n_products,
  concat_ws('; ', transform(b.product_issues, x -> concat(x.product, ': ', x.issue))) AS product_issues_str,
  size(b.product_issues)                       AS n_product_issues,
  b._review_uid                                AS review_uid
FROM base b
-- posexplode so pos=0 marks the primary row; reviews with no classification still
-- appear once (category NULL, pos NULL) via OUTER.
LATERAL VIEW OUTER posexplode(b.classification) c AS pos, category
"""

print(GENIE_SQL)
spark.sql(GENIE_SQL)
print(f"\nWrote {GENIE_TBL}")

In [ ]:
# --- Column + table comments (Genie reads these as metadata) ----------------
COMMENTS = {
    "locationId":              "Google directory location id for the store (bigint). Use city/state for human labels.",
    "location_name":           "Store label as 'City, ST' (e.g. 'Chesapeake, VA').",
    "city":                    "Store city, parsed from location_name.",
    "state":                   "Two-letter US state code for the store, parsed from location_name. Use for state filters.",
    "review_date":             "Date the review was posted (DATE). Reviews span multiple years.",
    "rating":                  "Overall star rating the reviewer gave, 1-5 (double).",
    "sentiment":               "Overall AI sentiment of the review: Positive, Negative, Mixed, or Neutral.",
    "review_source":           "Where the review came from (e.g. GOOGLE).",
    "review_text":             "The review text (AI-cleaned comment, falling back to the raw review).",
    "category":                "Experience aspect this row is about: Speed, Cleanliness, Order Accuracy, Quality, or Service. One row per aspect the review mentions; NULL if the review mentioned none.",
    "category_rating":         "AI 1-5 rating for THIS row's category, or NULL when the review did not score that aspect. NULL means 'not discussed', not zero: exclude NULLs from averages.",
    "is_primary_category_row": "TRUE on exactly one row per review. Filter WHERE is_primary_category_row to count reviews or average the overall rating without double-counting the category fan-out.",
    "products":                "Comma-separated menu products mentioned in the review (e.g. 'Pizza, Cheese'); empty if none.",
    "n_products":              "Count of distinct products mentioned in the review.",
    "product_issues_str":      "Semicolon-separated 'Product: Issue' pairs the reviewer complained about (e.g. 'Pizza: Burnt; Pizza: Stale/Dry'); empty if none.",
    "n_product_issues":        "Count of product-issue complaints in the review.",
    "review_uid":              "Stable id grouping all category rows that belong to the same review.",
}

spark.sql(f"COMMENT ON TABLE {GENIE_TBL} IS "
          f"'Google guest reviews for LCE stores, AI-enriched and flattened for analytics. "
          f"One row per review x mentioned category (see is_primary_category_row to get review grain). "
          f"Sourced from {GOLD}.'")

for col, desc in COMMENTS.items():
    safe = desc.replace("'", "''")
    spark.sql(f"ALTER TABLE {GENIE_TBL} ALTER COLUMN {col} COMMENT '{safe}'")

print(f"Applied {len(COMMENTS)} column comments + table comment.")
display(spark.sql(f"DESCRIBE TABLE {GENIE_TBL}"))

In [ ]:
# --- Validate the table -----------------------------------------------------
from pyspark.sql import functions as F

g = spark.table(GENIE_TBL)
print(f"rows (review x category): {g.count()}")
print(f"reviews (primary rows):   {g.filter('is_primary_category_row').count()}")
print(f"stores:                   {g.select('locationId').distinct().count()}")

# avg rating by category (NULL-aware) — the core analytics pattern Genie will mirror.
display(
    g.filter(F.col("category_rating").isNotNull())
     .groupBy("category")
     .agg(F.round(F.avg("category_rating"), 2).alias("avg_rating"), F.count("*").alias("n"))
     .orderBy("avg_rating")
)

## 2 · Create the Genie space

Builds a `serialized_space` (v2) wired to `reviews_gold_genie` with column descriptions +
synonyms, text instructions, curated example-question SQL, and benchmark questions, then
creates the space via `POST /api/2.0/genie/spaces`.

> Idempotency: this creates a NEW space each run. Re-running makes another space. Capture the
> returned `space_id` (printed + saved to a task value) and reuse it; delete stray spaces in the UI.
> To point the LCE app at this space, set `GENIE_SPACE_ID` in the app's `app.yaml` to the id below.

In [ ]:
# --- Identity + parent path (SDK only) --------------------------------------
# The notebook CommandContext is blocklisted on serverless / high-security compute,
# so use the auto-authenticated WorkspaceClient for host, identity, and API calls.
import json
from databricks.sdk import WorkspaceClient

w    = WorkspaceClient()
HOST = w.config.host  # e.g. https://adb-....azuredatabricks.net
try:
    ME = w.current_user.me().user_name
except Exception:
    ME = ""

parent_path = PARENT or (f"/Workspace/Users/{ME}" if ME else "/Workspace/Shared")
print(f"host: {HOST}\nuser: {ME or '(unknown)'}\nparent: {parent_path}")

In [ ]:
# --- 3 & 4: column descriptions/synonyms, instructions, example SQL, benchmarks ---
# Deterministic ids (no Math.random in a notebook run); Genie accepts any unique ids.
def _id(n):
    return f"cfg{n:032d}"

COL_CFG = [
    {"column_name": "city",            "enable_format_assistance": True, "enable_entity_matching": True,
     "synonyms": ["store city", "location city"]},
    {"column_name": "state",           "enable_format_assistance": True, "enable_entity_matching": True,
     "synonyms": ["US state", "state code"]},
    {"column_name": "location_name",   "enable_format_assistance": True, "enable_entity_matching": True,
     "synonyms": ["store", "restaurant", "location"]},
    {"column_name": "review_date",     "enable_format_assistance": True,
     "synonyms": ["date", "posted date", "review day"]},
    {"column_name": "rating",          "enable_format_assistance": True,
     "synonyms": ["stars", "star rating", "overall rating"]},
    {"column_name": "sentiment",       "enable_format_assistance": True, "enable_entity_matching": True,
     "synonyms": ["tone", "positivity"]},
    {"column_name": "category",        "enable_format_assistance": True, "enable_entity_matching": True,
     "synonyms": ["aspect", "experience category", "theme"]},
    {"column_name": "category_rating", "enable_format_assistance": True,
     "synonyms": ["aspect rating", "category score"]},
    {"column_name": "products",        "enable_format_assistance": True, "enable_entity_matching": True,
     "synonyms": ["menu items", "items mentioned"]},
    {"column_name": "product_issues_str", "enable_format_assistance": True,
     "synonyms": ["product complaints", "food issues"]},
]
# attach the in-table comments as descriptions too (belt and suspenders for Genie)
for c in COL_CFG:
    d = COMMENTS.get(c["column_name"])
    if d:
        c["description"] = [d]

INSTRUCTIONS = [
    "This space answers questions about Little Caesars guest reviews (Google reviews), "
    "AI-enriched with sentiment, per-aspect ratings, product mentions, and product complaints.",
    "Grain: one row per review x mentioned category. To count reviews or average the overall "
    "`rating`, filter `WHERE is_primary_category_row` so the category fan-out does not double-count.",
    "`category_rating` is NULL when a review did not score that aspect. NULL means 'not discussed', "
    "not zero: always exclude NULLs from category-rating averages.",
    "Categories are exactly: Speed, Cleanliness, Order Accuracy, Quality, Service. Sentiment is "
    "Positive, Negative, Mixed, or Neutral. Use `state` (2-letter) and `city` for geography.",
    "For 'worst/best stores', rank by share of Negative reviews or by average rating; always "
    "show the store count behind a metric so small samples are visible.",
]

def sql_lines(s):
    return [ln + "\n" for ln in s.strip().split("\n")]

EXAMPLE_SQLS = [
    {"q": "Which stores have the highest share of negative reviews?",
     "sql": f"""SELECT city, state,
       COUNT(*) AS reviews,
       ROUND(AVG(CASE WHEN sentiment = 'Negative' THEN 1.0 ELSE 0 END) * 100, 1) AS pct_negative
FROM {GENIE_TBL}
WHERE is_primary_category_row
GROUP BY city, state
HAVING COUNT(*) >= 5
ORDER BY pct_negative DESC
LIMIT 20"""},
    {"q": "What is the average rating for each category?",
     "sql": f"""SELECT category,
       ROUND(AVG(category_rating), 2) AS avg_rating,
       COUNT(*) AS scored_reviews
FROM {GENIE_TBL}
WHERE category_rating IS NOT NULL
GROUP BY category
ORDER BY avg_rating ASC"""},
    {"q": "Which products get the most complaints and what are the issues?",
     "sql": f"""SELECT product_issues_str, COUNT(*) AS mentions
FROM {GENIE_TBL}
WHERE is_primary_category_row AND n_product_issues > 0
GROUP BY product_issues_str
ORDER BY mentions DESC
LIMIT 20"""},
]

BENCHMARKS = [
    "Which stores have the most negative reviews?",
    "What is the average rating by category across all stores?",
    "Show cleanliness ratings by state.",
    "What products get the most complaints?",
    "How has sentiment trended over the last 3 months?",
]

serialized = {
    "version": 2,
    "config": {
        "sample_questions": [{"id": _id(i), "question": [q]} for i, q in enumerate(BENCHMARKS)]
    },
    "data_sources": {
        "tables": [{"identifier": GENIE_TBL, "column_configs": COL_CFG}]
    },
    "instructions": {
        "text_instructions": [{"id": _id(100 + i), "content": [t]} for i, t in enumerate(INSTRUCTIONS)],
        "example_question_sqls": [
            {"id": _id(200 + i), "question": [e["q"]], "sql": sql_lines(e["sql"])}
            for i, e in enumerate(EXAMPLE_SQLS)
        ],
    },
}
print(json.dumps(serialized, indent=2)[:1500])

In [ ]:
# --- Create the space (SDK api_client, no raw requests/token) ---------------
payload = {
    "title": TITLE,
    "description": "Natural-language analytics over LCE Google guest reviews "
                   "(sentiment, per-aspect ratings, product complaints). Backed by "
                   f"{GENIE_TBL}.",
    "parent_path": parent_path,
    "warehouse_id": WAREHOUSE,
    "serialized_space": json.dumps(serialized),
}

body = w.api_client.do("POST", "/api/2.0/genie/spaces", body=payload)
print(json.dumps(body, indent=2)[:800])

space_id = body.get("space_id") or body.get("id")
if space_id:
    print(f"\n✅ Created Genie space: {space_id}")
    print(f"   {HOST}/genie/rooms/{space_id}")
    print(f"   To wire the app: set GENIE_SPACE_ID = {space_id} in app.yaml, then redeploy.")
else:
    raise RuntimeError(f"Space creation did not return an id: {body}")

In [ ]:
# --- Round-trip verify (SDK api_client) -------------------------------------
chk = w.api_client.do(
    "GET", f"/api/2.0/genie/spaces/{space_id}",
    query={"include_serialized_space": "true"},
)
ss = json.loads(chk["serialized_space"]) if isinstance(chk.get("serialized_space"), str) else {}
tables = [t["identifier"] for t in ss.get("data_sources", {}).get("tables", [])]
print("title:       ", chk.get("title"))
print("warehouse:   ", chk.get("warehouse_id"))
print("tables:      ", tables)
print("instructions:", len(ss.get("instructions", {}).get("text_instructions", [])))
print("example sqls:", len(ss.get("instructions", {}).get("example_question_sqls", [])))
print("benchmarks:  ", len(ss.get("config", {}).get("sample_questions", [])))